## Step 2 - Cut keyframes + build metadata

Reads the shot boundaries from Step 1 and extracts representative frames
(first / middle / last of each shot). Produces the **nested** layout the backend
expects and the metadata file it reads:

```
Keyframes/<batch>/<video_name>/<idx:06d>.jpg   e.g. Keyframes/L21/L21_V001/000080.jpg
frames_meta.json  = { "<video_name>_<idx:06d>": {video_name, frame_name, timestamp_ms, fps} }
```

`video_name = <batch>_<video_id>` (e.g. `L21_V001`); `frame_id = <video_name>_<idx>`
(e.g. `L21_V001_000080`) is the join key used by the FAISS id_map and OCR files.
`batch` is recoverable as `video_name.split('_')[0]` (e.g. `L21`).

> `frame_name` is recorded with the **final** extension `FINAL_EXT` (default `webp`)
> because Step 3 re-encodes the jpg keyframes to webp. Keep them in sync.

> **Khử trùng lặp (cosine):** `DEDUP_THRESHOLD` (mặc định `0.93`) bỏ các frame
> gần-giống ngay khi cắt - so cosine trên pixel (ảnh xám 32×32, trừ trung bình)
> với frame giữ gần nhất *trong cùng video*. Nhờ đó đĩa/embedding/index đều nhẹ
> hơn. Đặt `DEDUP_THRESHOLD = 0` để tắt (giữ đủ first/mid/last mọi shot).


In [18]:
import os
os.environ["OPENCV_FFMPEG_LOGLEVEL"] = "-8"   # silence noisy H.264 decoder warnings
import cv2
import json
from tqdm import tqdm

In [ ]:
VIDEOS_DIR    = '../../backend/media/videos'                 # Videos_<batch>/video/*.mp4
SCENE_JSON_DIR = '../../backend/media/scenes/SceneJSON'       # output of Step 1
SAVE_DIR      = './Keyframes'
META_PATH     = './frames_meta.json'
FINAL_EXT     = 'webp'    # extension the images will have AFTER Step 3 (reduce)

# Khử trùng lặp gần-giống bằng cosine trên pixel frame (chưa có embedding ở bước này).
# So mỗi frame ứng viên với frame giữ gần nhất TRONG CÙNG video; > ngưỡng -> bỏ.
# 0 = tắt. 0.90-0.95 hợp lý; cao hơn = bỏ ít hơn. Chỉnh tùy độ 'tĩnh' của cảnh.
DEDUP_THRESHOLD = 0.95

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Quét đệ quy MỌI file video, không phụ thuộc cách chia thư mục
# (Videos_L21/video/..., Videos_L25_a/..., Videos_L25_b/..., lồng sâu tùy ý).
# video_name = tên file (stem), ví dụ L25_V001.mp4 -> video_name 'L25_V001',
# batch 'L25', video_id 'V001'. Khớp key join của các bước sau.
#
# GỘP THEO TÊN FILE, KHÔNG THEO FOLDER: các folder Videos_L25_a / _a1 / _b đều
# gộp về batch 'L25' vì key lấy từ TÊN FILE. Vài dataset upload LẶP một part
# (vd Videos_L25_a và Videos_L25_a1 chứa cùng dải L25_V001..V088). Ta phân biệt:
#   * cùng tên + CÙNG dung lượng  -> bản sao dư thừa, bỏ qua (giữ bản gặp đầu).
#   * cùng tên + KHÁC dung lượng  -> hai video khác nội dung xài chung id -> DỪNG,
#     vì gộp về cùng batch sẽ mất data (cần đưa hậu tố folder vào khóa).
from pathlib import Path
VIDEO_EXTS = ('.mp4', '.mkv', '.webm', '.avi', '.mov')

all_video_paths = {}
seen = {}                 # video_name -> (path, size) của bản gặp đầu tiên
conflicts = []            # trùng tên NHƯNG khác nội dung (khác size)
redundant = 0             # trùng tên VÀ cùng size -> bản sao, bỏ qua an toàn
skipped = 0
for vid in sorted(Path(VIDEOS_DIR).rglob('*')):
    if not (vid.is_file() and vid.suffix.lower() in VIDEO_EXTS):
        continue
    video_name = vid.stem                              # 'L25_V001'
    if '_' not in video_name:                          # tên không có batch -> bỏ
        skipped += 1
        continue
    if video_name in seen:
        prev_path, prev_size = seen[video_name]
        if vid.stat().st_size == prev_size:            # cùng tên + cùng size -> bản sao
            redundant += 1
        else:                                          # cùng tên, khác nội dung -> xung đột
            conflicts.append((video_name, prev_path, str(vid)))
        continue
    seen[video_name] = (str(vid), vid.stat().st_size)
    batch, video_id = video_name.split('_', 1)         # 'L25', 'V001'
    all_video_paths.setdefault(batch, {})[video_id] = str(vid)

print('batches:', {k: len(v) for k, v in all_video_paths.items()})
if skipped:
    print(f'[WARN] bỏ qua {skipped} file không có batch trong tên (dạng L25_V001 mới hợp lệ)')
if redundant:
    print(f'[WARN] bỏ qua {redundant} file trùng-lặp (cùng tên & cùng dung lượng ở '
          'folder khác, vd Videos_L25_a vs Videos_L25_a1) -> giữ bản gặp đầu tiên.')
if conflicts:
    print(f'[LỖI] {len(conflicts)} tên video trùng NHƯNG khác nội dung (khác dung lượng):')
    for name, a, b in conflicts[:20]:
        print(f'  {name}: {a}  <->  {b}')
    raise SystemExit(
        f'{len(conflicts)} video khác nội dung dùng chung tên -> video_id không '
        'duy nhất trong batch. Cần đưa hậu tố folder (_a/_b) vào khóa và sửa đồng '
        'bộ các bước sau. Xem danh sách ở trên.'
    )


In [21]:
def sample_frames_from_shot(start_idx, end_idx):
    """First / middle / last frame of a shot (matches the AIC-HCMC-2025 heuristic)."""
    mid = start_idx + (end_idx - start_idx) // 2
    return sorted({start_idx, mid, end_idx})

In [22]:
frames_meta = {}
if os.path.exists(META_PATH):
    with open(META_PATH, 'r', encoding='utf-8') as f:
        frames_meta = json.load(f)


def frame_signature(frame):
    """Chữ ký nhẹ để so gần-trùng: ảnh xám 32x32, trừ trung bình, L2-normalize.
    Dot của 2 chữ ký = tương quan (cosine) trong [-1, 1]. Trừ trung bình giúp
    ít nhạy với thay đổi độ sáng tổng thể hơn là cosine trên pixel thô."""
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA).astype('float32').reshape(-1)
    g -= g.mean()
    return g / (float(g @ g) ** 0.5 + 1e-9)


kept = dropped = 0
for batch, videos in all_video_paths.items():
    for video_id in tqdm(sorted(videos), desc=batch):
        video_name = f'{batch}_{video_id}'                 # flat: L21_V001
        scene_path = f'{SCENE_JSON_DIR}/{batch}/{video_id}.json'
        if not os.path.exists(scene_path):
            print('missing scenes:', scene_path)
            continue
        with open(scene_path, 'r') as f:
            scenes = json.load(f)

        out_dir = f'{SAVE_DIR}/{batch}/{video_name}'      # nested: Keyframes/L21/L21_V001
        os.makedirs(out_dir, exist_ok=True)

        # All frame indices we want for this video (first/mid/last of each shot)
        wanted = sorted({
            idx
            for shot in scenes
            for idx in sample_frames_from_shot(int(shot[0]), int(shot[1]))
        })

        cap = cv2.VideoCapture(videos[video_id])
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

        # Sequential decode: read frames in order and pick the wanted ones.
        # Avoids H.264 random seeking (cap.set(POS_FRAMES, ...)), which is not
        # frame-accurate and triggers the "mmco: unref short failure" warnings /
        # decode artifacts when landing mid-GOP without reference frames.
        todo = set(wanted)
        max_idx = wanted[-1] if wanted else -1
        idx = 0
        last_sig = None          # reset dedup theo từng video
        while todo and idx <= max_idx:
            ret, frame = cap.read()
            if not ret:
                break
            if idx in todo:
                todo.discard(idx)
                # Bỏ frame gần-trùng với frame giữ gần nhất trong video này
                if DEDUP_THRESHOLD > 0:
                    sig = frame_signature(frame)
                    if last_sig is not None and float(sig @ last_sig) > DEDUP_THRESHOLD:
                        dropped += 1
                        idx += 1
                        continue
                    last_sig = sig
                fpath = f'{out_dir}/{idx:06d}.jpg'
                if os.path.exists(fpath) or cv2.imwrite(fpath, frame):
                    kept += 1
                    frame_id = f'{video_name}_{idx:06d}'
                    frames_meta[frame_id] = {
                        'video_name': video_name,
                        'frame_name': f'{idx:06d}.{FINAL_EXT}',
                        'timestamp_ms': int(idx / fps * 1000),
                        'fps': float(fps),
                    }
            idx += 1
        cap.release()

with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(frames_meta, f, ensure_ascii=False, indent=2)
print(f'frames kept: {kept}, dropped near-dup: {dropped} (cosine > {DEDUP_THRESHOLD}) -> {META_PATH}')


L22: 100%|██████████| 31/31 [11:15<00:00, 21.79s/it]

frames: 54955 -> ./frames_meta.json
